##Aim:
To design and implement an ETL pipeline that extracts data from CSV and JSON files, cleans and validates the data, removes invalid records, transforms the data, and loads it incrementally into a relational database.

In [ ]:
# ============================================================
# Practical No. 7: ETL Pipeline
# ============================================================

import pandas as pd
import json
import sqlite3
import os


# ------------------------------------------------------------
# Step 1: Create Sample CSV Files
# ------------------------------------------------------------

csv1 = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "age": [22, 25, 30],
    "city": ["Mumbai", "Delhi", "Pune"]
})

csv2 = pd.DataFrame({
    "id": [4, 5, 6],
    "name": ["David", "Eva", "Frank"],
    "age": [28, -5, 35],       # -5 is invalid
    "city": ["Nashik", "Mumbai", "Nagpur"]
})

csv1.to_csv("customers1.csv", index=False)
csv2.to_csv("customers2.csv", index=False)


# ------------------------------------------------------------
# Step 2: Extract Data from Multiple CSV Files
# ------------------------------------------------------------

df1 = pd.read_csv("customers1.csv")
df2 = pd.read_csv("customers2.csv")

# Combine both CSV datasets
csv_data = pd.concat([df1, df2], ignore_index=True)

print("Combined CSV Data:")
print(csv_data)


# ------------------------------------------------------------
# Step 3: Clean and Remove Invalid Records
# ------------------------------------------------------------

# Remove duplicate records
csv_data.drop_duplicates(inplace=True)

# Remove records with invalid age
clean_data = csv_data[
    (csv_data["age"] >= 18) &
    (csv_data["age"] <= 100)
].copy()

print("\nData After Removing Invalid Records:")
print(clean_data)


# ------------------------------------------------------------
# Step 4: Data Transformation
# ------------------------------------------------------------

# Convert names to uppercase
clean_data["name"] = clean_data["name"].str.upper()

# Create a new transformed field
clean_data["category"] = clean_data["age"].apply(
    lambda x: "Young" if x < 30 else "Adult"
)

print("\nTransformed Data:")
print(clean_data)


# ------------------------------------------------------------
# Step 5: Create JSON Data
# ------------------------------------------------------------

json_data = [
    {"id": 7, "name": "Grace", "age": 24, "city": "Thane"},
    {"id": 8, "name": "Henry", "age": 32, "city": "Mumbai"}
]

with open("customers.json", "w") as file:
    json.dump(json_data, file)


# ------------------------------------------------------------
# Step 6: Extract and Transform JSON Data
# ------------------------------------------------------------

with open("customers.json", "r") as file:
    json_records = json.load(file)

json_df = pd.DataFrame(json_records)

# Transform selected fields
json_df["name"] = json_df["name"].str.upper()
json_df["category"] = json_df["age"].apply(
    lambda x: "Young" if x < 30 else "Adult"
)

print("\nJSON Data After Transformation:")
print(json_df)


# ------------------------------------------------------------
# Step 7: Combine CSV and JSON Data
# ------------------------------------------------------------

final_data = pd.concat(
    [clean_data, json_df],
    ignore_index=True
)

print("\nFinal ETL Dataset:")
print(final_data)


# ------------------------------------------------------------
# Step 8: Data Validation
# ------------------------------------------------------------

# Check required fields and valid age values
valid_data = final_data[
    final_data["id"].notna() &
    final_data["name"].notna() &
    final_data["age"].between(18, 100)
].copy()

print("\nValidated Data:")
print(valid_data)


# ------------------------------------------------------------
# Step 9: Connect to Relational Database
# ------------------------------------------------------------

connection = sqlite3.connect("etl_database.db")


# ------------------------------------------------------------
# Step 10: Incremental Loading
# ------------------------------------------------------------

# Create target table if it does not exist
connection.execute("""
CREATE TABLE IF NOT EXISTS customers (
    id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    city TEXT,
    category TEXT
)
""")

# Find the latest ID already stored in the database
result = connection.execute(
    "SELECT MAX(id) FROM customers"
).fetchone()

last_id = result[0] if result[0] is not None else 0

# Load only new records
new_records = valid_data[
    valid_data["id"] > last_id
]

if not new_records.empty:
    new_records.to_sql(
        "customers",
        connection,
        if_exists="append",
        index=False
    )

    print("\nNew records loaded successfully:")
    print(new_records)
else:
    print("\nNo new records to load.")


# ------------------------------------------------------------
# Step 11: Verify Data in Database
# ------------------------------------------------------------

database_data = pd.read_sql(
    "SELECT * FROM customers",
    connection
)

print("\nFinal Data in Database:")
print(database_data)

connection.close()

print("\nETL Pipeline completed successfully!")

Combined CSV Data:
   id     name  age    city
0   1    Alice   22  Mumbai
1   2      Bob   25   Delhi
2   3  Charlie   30    Pune
3   4    David   28  Nashik
4   5      Eva   -5  Mumbai
5   6    Frank   35  Nagpur

Data After Removing Invalid Records:
   id     name  age    city
0   1    Alice   22  Mumbai
1   2      Bob   25   Delhi
2   3  Charlie   30    Pune
3   4    David   28  Nashik
5   6    Frank   35  Nagpur

Transformed Data:
   id     name  age    city category
0   1    ALICE   22  Mumbai    Young
1   2      BOB   25   Delhi    Young
2   3  CHARLIE   30    Pune    Adult
3   4    DAVID   28  Nashik    Young
5   6    FRANK   35  Nagpur    Adult

JSON Data After Transformation:
   id   name  age    city category
0   7  GRACE   24   Thane    Young
1   8  HENRY   32  Mumbai    Adult

Final ETL Dataset:
   id     name  age    city category
0   1    ALICE   22  Mumbai    Young
1   2      BOB   25   Delhi    Young
2   3  CHARLIE   30    Pune    Adult
3   4    DAVID   28  Nashik    Y

Urvi Sawant | 42